# EB Causal SPINN Forward Problem (Demo)

Euler-Bernoulli beam, 16th vibration mode.

PDE: $u_{tt} + u_{xxxx} + (\pi^2 - 1)\,u = 0$, domain $x \in [0, 16\pi]$, $t \in [0,1]$.

Standalone notebook: inlines the SPINN model, autodiff helper, data
generator, loss, and training loop. No imports from `networks/` or
`utils/` are required.


In [ ]:
import jax
jax.config.update('jax_default_matmul_precision', 'float32')

import jax.numpy as jnp
import numpy as np
import optax
from jax import jvp, value_and_grad
from flax import linen as nn
from typing import Sequence
from functools import partial
from tqdm.auto import trange
import matplotlib.pyplot as plt


## Hyperparameters

In [ ]:
SEED       = 111
NC         = 128                # collocation points per axis (also IC/BC)
NC_TEST    = 100
LR         = 1e-5
EPOCHS     = 5_000              # bump to 250_000 for paper-quality results
N_LAYERS   = 5
FEATURES   = 128
R          = 128                # tensor rank
X_MIN, X_MAX = 0.0, 16 * np.pi
T_MAX      = 1.0


## SPINN model + HVP

In [ ]:
# Forward-over-forward HVP.  Used to compute u_xx, u_tt, u_xxxx, etc.
def hvp_fwdfwd(f, primals, tangents, return_primals=False):
    g = lambda primals: jvp(f, (primals,), tangents)[1]
    primals_out, tangents_out = jvp(g, primals, tangents)
    if return_primals:
        return primals_out, tangents_out
    return tangents_out


In [ ]:
# Separable PINN with 2 axes (t, x).  One modified MLP per axis whose final
# outputs (rank r) are merged via outer product: out = U_t @ U_x^T.
class SPINN2d(nn.Module):
    features: Sequence[int]    # hidden widths, last entry is the layer before rank
    r: int                     # tensor rank
    mlp: str = 'modified_mlp'

    @nn.compact
    def __call__(self, t, x):
        inputs, outputs = [t, x], []
        init = nn.initializers.glorot_normal()
        for X in inputs:
            if self.mlp == 'mlp':
                for fs in self.features[:-1]:
                    X = nn.tanh(nn.Dense(fs, kernel_init=init)(X))
                X = nn.Dense(self.r, kernel_init=init)(X)
            else:  # modified_mlp (gated)
                U = nn.tanh(nn.Dense(self.features[0], kernel_init=init)(X))
                V = nn.tanh(nn.Dense(self.features[0], kernel_init=init)(X))
                H = nn.tanh(nn.Dense(self.features[0], kernel_init=init)(X))
                for fs in self.features[:-1]:
                    Z = nn.tanh(nn.Dense(fs, kernel_init=init)(H))
                    H = (1 - Z) * U + Z * V
                X = nn.Dense(self.r, kernel_init=init)(H)
            outputs.append(X)
        # outputs[0]: (T, r),  outputs[1]: (Nx, r)  →  (T, Nx)
        return jnp.dot(outputs[0], outputs[1].T)


## Analytic solution & data generator

In [ ]:
def exact_u(t, x):
    # u(t, x) = sin(x) * cos(pi t).  Matches the 16th-mode setup
    # (sin(16π) = 0 gives zero displacement at both ends).
    return jnp.sin(x) * jnp.cos(jnp.pi * t)

def source_term(t, x):
    return 0.0  # u_tt = -π² u, u_xxxx = u  →  residual vanishes

def make_train_data(nc):
    tc = jnp.linspace(0, T_MAX, nc + 2)[1:-1].reshape(-1, 1)
    xc = jnp.linspace(X_MIN, X_MAX, nc + 2)[1:-1].reshape(-1, 1)
    tm, xm = jnp.meshgrid(tc.ravel(), xc.ravel(), indexing='ij')
    uc = jnp.broadcast_to(source_term(tm, xm), tm.shape)

    # Initial condition at t=0
    ti = jnp.zeros((1, 1))
    xi = xc
    ti_m, xi_m = jnp.meshgrid(ti.ravel(), xi.ravel(), indexing='ij')
    ui = exact_u(ti_m, xi_m)

    # Boundaries at x = 0 and x = 16π
    tb  = tc
    xbl = jnp.zeros((nc, 1))
    xbr = X_MAX * jnp.ones((nc, 1))
    tbl_m, xbl_m = jnp.meshgrid(tb.ravel(), xbl.ravel(), indexing='ij')
    tbr_m, xbr_m = jnp.meshgrid(tb.ravel(), xbr.ravel(), indexing='ij')
    ubl = exact_u(tbl_m, xbl_m)
    ubr = exact_u(tbr_m, xbr_m)

    # Lower-triangular causal mask: aggregates loss from earlier time steps
    W = jnp.tril(jnp.ones((nc, nc)), k=-1)
    return tc, xc, uc, ti, xi, ui, tb, xbl, xbr, ubl, ubr, W


## Causal loss

$$
\mathcal{L}_{\text{res}}(t_i) =
\exp\!\Big(-\varepsilon \sum_{j<i} \mathcal{L}_{\text{res}}(t_j)\Big) \,
\mathcal{L}_{\text{res}}(t_i)
$$


In [ ]:
@partial(jax.jit, static_argnames=('apply_fn',))
def loss_and_grad(apply_fn, params, *train_data):
    tc, xc, uc, ti, xi, ui, tb, xbl, xbr, ubl, ubr, W = train_data

    def residual_loss(p):
        u = apply_fn(p, tc, xc)
        v = jnp.ones(tc.shape)
        u_tt   = hvp_fwdfwd(lambda t: apply_fn(p, t, xc), (tc,), (v,))
        u_xxxx = hvp_fwdfwd(
            lambda x: hvp_fwdfwd(lambda x: apply_fn(p, tc, x), (x,), (v,)),
            (xc,), (v,))
        res = u_tt + u_xxxx + (jnp.pi**2 - 1) * u - uc
        loss_time = jnp.mean(res**2, axis=1, keepdims=True)         # (T, 1)
        agg = jax.lax.stop_gradient(jnp.dot(W, loss_time))
        causal_w = jnp.exp(-5.0 * agg)
        return jnp.mean(causal_w * loss_time)

    def initial_loss(p):
        ic_disp = jnp.mean((apply_fn(p, ti, xi) - ui)**2)
        v_t = jnp.ones(ti.shape)
        u_t0 = jvp(lambda t: apply_fn(p, t, xi), (ti,), (v_t,))[1]
        ic_vel = jnp.mean(u_t0**2)
        return ic_disp + ic_vel

    def boundary_loss(p):
        v = jnp.ones(tb.shape)
        ul = apply_fn(p, tb, xbl); ur = apply_fn(p, tb, xbr)
        uxx_l = hvp_fwdfwd(lambda xbl: apply_fn(p, tb, xbl), (xbl,), (v,))
        uxx_r = hvp_fwdfwd(lambda xbr: apply_fn(p, tb, xbr), (xbr,), (v,))
        return (jnp.mean((ul - ubl)**2) + jnp.mean((ur - ubr)**2)
              + jnp.mean((uxx_l - ubr)**2) + jnp.mean((uxx_r - ubl)**2))

    total = lambda p: 0.1 * residual_loss(p) + initial_loss(p) + boundary_loss(p)
    return value_and_grad(total)(params)


## Initialize and train

In [ ]:

key = jax.random.PRNGKey(SEED)
key, sub_init = jax.random.split(key, 2)

model = SPINN2d(features=[FEATURES] * N_LAYERS, r=R)
t0 = jnp.ones((NC, 1)); x0 = jnp.ones((NC, 1))
params = model.init(sub_init, t0, x0)
apply_fn = jax.jit(model.apply)

n_params = sum(x.size for x in jax.tree_util.tree_leaves(params))
print(f"Total trainable params: {n_params}")

train_data = make_train_data(NC)

# Eval grid + ground truth for periodic best-tracking
t_eval = jnp.linspace(0, T_MAX, 100).reshape(-1, 1)
x_eval = jnp.linspace(X_MIN, X_MAX, 100).reshape(-1, 1)
tm_eval, xm_eval = jnp.meshgrid(t_eval.ravel(), x_eval.ravel(), indexing='ij')
u_true_eval = exact_u(tm_eval, xm_eval)

optim = optax.adam(LR)
state = optim.init(params)

losses = []
best_loss = 1e9
best_err = 1e9
best_params = params

pbar = trange(EPOCHS)
for e in pbar:
    loss, grads = loss_and_grad(apply_fn, params, *train_data)
    updates, state = optim.update(grads, state, params)
    params = optax.apply_updates(params, updates)
    losses.append(float(loss))
    if float(loss) <= best_loss:                       
        best_loss = float(loss)
        u_pred_eval = apply_fn(params, t_eval, x_eval)
        best_err = float(jnp.linalg.norm(u_pred_eval - u_true_eval) / jnp.linalg.norm(u_true_eval))
        best_params = params                            # save unconditionally on best-loss step
    if (e + 1) % 500 == 0:
        pbar.set_postfix(loss=f"{loss:.3e}", err_at_best_loss=f"{best_err:.3e}")

print(f"\nRel L2 error at best-loss step: {best_err:.3e}")


## Evaluate and plot (using BEST params)

In [ ]:
# Relative L2 error against the analytic solution.
def relative_l2(pred, true):
    return float(jnp.linalg.norm(pred - true) / jnp.linalg.norm(true))

# Visualize predictions from the BEST-tracked params (not end-of-training)
t_test = jnp.linspace(0, T_MAX, 100).reshape(-1, 1)
x_test = jnp.linspace(X_MIN, X_MAX, 100).reshape(-1, 1)
tm, xm = jnp.meshgrid(t_test.ravel(), x_test.ravel(), indexing='ij')
u_true = exact_u(tm, xm)
u_pred = apply_fn(best_params, t_test, x_test)
err = best_err  # already computed during training
print(f"Best relative L2 error: {err:.3e}")

fig, axs = plt.subplots(1, 3, figsize=(15, 4))
for ax, data, title in zip(
        axs,
        [u_true, u_pred, jnp.abs(u_true - u_pred)],
        ['Exact $u(t,x)$', 'Predicted $\hat u(t,x)$', '|Error|']):
    im = ax.pcolormesh(np.asarray(tm), np.asarray(xm), np.asarray(data),
                       cmap='RdBu_r', shading='auto')
    ax.set_xlabel('t'); ax.set_ylabel('x'); ax.set_title(title)
    plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()


In [ ]:
# Loss curve
plt.figure(figsize=(8, 4))
plt.semilogy(losses)
plt.xlabel('epoch'); plt.ylabel('total loss')
plt.title('EB Causal SPINN — training loss')
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
